In [8]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [9]:
%tensorboard --logdir runs/vgg_imagenet_experiment

In [4]:
import json
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1) Synset ID → human-readable label 매핑 불러오기
with open("imagenet_class_index.json") as f:
    class_idx = json.load(f)
# class_idx 예시: {"0": ["n01440764","tench"], "1": ["n01443537","goldfish"], ...}

# 2) ImageFolder 로 데이터셋 생성
train_dataset = datasets.ImageFolder(
    root="D:/Changhee/data/ImageNet_train",
    transform=transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485,0.456,0.406],
            std=[0.229,0.224,0.225]
        ),
    ])
)

# 3) DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)

# 4) 인덱스 매핑 생성
idx_to_synset = {v: k for k, v in train_dataset.class_to_idx.items()}
# idx_to_synset: {0: "n01440764", 1: "n01443537", ...}

# ——— 테스트 ———

print("📌 데이터셋 정보 출력하기")
print(f"Total images: {len(train_dataset)}")
print(f"Total classes: {len(train_dataset.classes)}")
print("Classes sample:", train_dataset.classes[:10])

print("\n📌 맵핑 정보(sample) 출력하기")
print("class_to_idx:", train_dataset.class_to_idx)
print("idx_to_synset:", idx_to_synset)
print("1번 클래스 human name:", class_idx["1"][1])   # goldfish

print("\n📌 배치 하나 꺼내서 레이블, 이름 확인하기")
images, labels = next(iter(train_loader))
print("Batch image tensor shape:", images.shape)
print("Batch label indices:", labels.tolist())
names = [class_idx[str(l.item())][1] for l in labels]
print("Batch human names:", names)

print("\n📌 개별 샘플 하나만 확인하기")
img_path, label = train_dataset.samples[0]
print("First image path:", img_path)
print("Label idx:", label)
print("Synset ID:", idx_to_synset[label])
print("Human name:", class_idx[str(label)][1])


📌 데이터셋 정보 출력하기
Total images: 1280379
Total classes: 1000
Classes sample: ['n01440764', 'n01443537', 'n01484850', 'n01491361', 'n01494475', 'n01496331', 'n01498041', 'n01514668', 'n01514859', 'n01518878']

📌 맵핑 정보(sample) 출력하기
class_to_idx: {'n01440764': 0, 'n01443537': 1, 'n01484850': 2, 'n01491361': 3, 'n01494475': 4, 'n01496331': 5, 'n01498041': 6, 'n01514668': 7, 'n01514859': 8, 'n01518878': 9, 'n01530575': 10, 'n01531178': 11, 'n01532829': 12, 'n01534433': 13, 'n01537544': 14, 'n01558993': 15, 'n01560419': 16, 'n01580077': 17, 'n01582220': 18, 'n01592084': 19, 'n01601694': 20, 'n01608432': 21, 'n01614925': 22, 'n01616318': 23, 'n01622779': 24, 'n01629819': 25, 'n01630670': 26, 'n01631663': 27, 'n01632458': 28, 'n01632777': 29, 'n01641577': 30, 'n01644373': 31, 'n01644900': 32, 'n01664065': 33, 'n01665541': 34, 'n01667114': 35, 'n01667778': 36, 'n01669191': 37, 'n01675722': 38, 'n01677366': 39, 'n01682714': 40, 'n01685808': 41, 'n01687978': 42, 'n01688243': 43, 'n01689811': 44, 'n01

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter


def main():
    # 설정
    data_dir = 'D:/Changhee/data'
    train_dir = os.path.join(data_dir, 'ImageNet_train')
    val_dir = os.path.join(data_dir, 'ImageNet_test')
    batch_size = 64
    num_epochs = 20
    learning_rate = 0.01
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # TensorBoard SummaryWriter
    writer = SummaryWriter(log_dir='runs/vgg_imagenet_experiment')

    # 데이터 변환
    train_transforms = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])
    val_transforms = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])

    # 데이터셋 및 데이터로더
    train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
    val_dataset = datasets.ImageFolder(root=val_dir, transform=val_transforms)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=8)

    # 모델 준비 (VGG16)
    model = models.vgg16(num_classes=len(train_dataset.classes))
    model = model.to(device)

    # 손실함수 및 옵티마이저
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    # 모델 구조를 TensorBoard에 기록
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    writer.add_graph(model, dummy_input)

    global_step = 0
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            global_step += 1

            # 배치 스케일 로그
            if global_step % 100 == 0:
                writer.add_scalar('Train/Batch_Loss', loss.item(), global_step)

        # 에폭별 통계
        epoch_loss = running_loss / len(train_dataset)
        epoch_acc = running_corrects.double() / len(train_dataset)
        writer.add_scalar('Train/Epoch_Loss', epoch_loss, epoch)
        writer.add_scalar('Train/Epoch_Acc', epoch_acc, epoch)

        # 검증 단계
        model.eval()
        val_loss = 0.0
        val_corrects = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data)

        val_loss = val_loss / len(val_dataset)
        val_acc = val_corrects.double() / len(val_dataset)
        writer.add_scalar('Val/Loss', val_loss, epoch)
        writer.add_scalar('Val/Acc', val_acc, epoch)

        print(f'Epoch {epoch+1}/{num_epochs}  '
              f'Train Loss: {epoch_loss:.4f}  Train Acc: {epoch_acc:.4f}  '
              f'Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}')

        scheduler.step()

    writer.close()


if __name__ == '__main__':
    main()
